In [ ]:
-- =============================================================================
-- Script: transform_userprofile_raw_data
-- Bronze → Bronze: route Debezium CDC events to per-table Delta tables
-- Source:  lh_vv_bronze.dbo.userprofile_raw_data  (Debezium envelope)
-- Targets: lh_vv_bronze.partner.{travel_profiles, user_plans, user_plan_itineraries, user_plan_itinerary_items, users}
-- Run:     Execute sequentially in a Fabric notebook (%%sql per block) or
--          via spark.sql() after each semicolon
--
-- CDC op mapping:
--   c / r / u  →  MERGE INTO target (upsert); deleted = FALSE
--   d          →  MERGE INTO target SET deleted = TRUE (soft delete; row is kept)
--
-- Deduplication:
--   Each upsert subquery deduplicates by (id, ts_ms DESC) via ROW_NUMBER so that
--   only the latest CDC event per key is applied — prevents MERGE ambiguity when
--   multiple events for the same id exist in the batch.
-- =============================================================================

SET spark.sql.legacy.timeParserPolicy=LEGACY;

-- =============================================================================
-- 0. DDL - TẠO CÁC BẢNG ĐÍCH TRONG SCHEMA lh_vv_bronze.partner
-- =============================================================================

CREATE SCHEMA IF NOT EXISTS lh_vv_bronze.partner;

-- Bảng travel_profiles
CREATE TABLE IF NOT EXISTS lh_vv_bronze.partner.travel_profiles (
    id STRING NOT NULL,
    created_at BIGINT,
    updated_at BIGINT,
    is_active BOOLEAN,
    user_id STRING,
    persona STRING,
    travel_dna STRING,
    travel_styles STRING,
    dietary_preferences STRING,
    accommodation_amenities_preferences STRING,
    metadata STRING,
    deleted BOOLEAN,
    _ingested_at TIMESTAMP, 
    _source_db STRING
)
USING DELTA
CLUSTER BY (id, user_id)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

-- Bảng user_plans
CREATE TABLE IF NOT EXISTS lh_vv_bronze.partner.user_plans (
    id STRING NOT NULL,
    creator_id STRING,
    title STRING NOT NULL,
    start_date BIGINT,
    end_date BIGINT,
    estimated_total_person INT,
    estimated_total_budget STRING,
    estimated_budgets STRING,
    currency STRING,
    weather_forecast STRING,
    traffic_suggestions STRING,
    trip_id STRING,
    total_days INT,
    share_code STRING,
    created_at BIGINT,
    updated_at BIGINT,
    metadata STRING,
    deleted BOOLEAN,
    _ingested_at TIMESTAMP, 
    _source_db STRING
)
USING DELTA
CLUSTER BY (id, creator_id)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

-- Bảng user_plan_itineraries
CREATE TABLE IF NOT EXISTS lh_vv_bronze.partner.user_plan_itineraries (
    id STRING NOT NULL,
    plan_id STRING NOT NULL,
    day_number INT,
    `date` BIGINT,
    day_type STRING NOT NULL,
    weather_forecast STRING,
    traffic_suggestion STRING,
    created_at BIGINT,
    updated_at BIGINT,
    metadata STRING,
    deleted BOOLEAN,
    _ingested_at TIMESTAMP, 
    _source_db STRING
)
USING DELTA
CLUSTER BY (plan_id, id)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

-- Bảng user_plan_itinerary_items
CREATE TABLE IF NOT EXISTS lh_vv_bronze.partner.user_plan_itinerary_items (
    id STRING NOT NULL,
    plan_itinerary_id STRING NOT NULL,
    `rank` INT NOT NULL,
    itinerary_type STRING NOT NULL,
    item_id STRING,
    item_metadata STRING,
    suggestion_note STRING,
    category_key STRING,
    selection_reason STRING,
    created_at BIGINT,
    updated_at BIGINT,
    metadata STRING,
    deleted BOOLEAN,
    _ingested_at TIMESTAMP, 
    _source_db STRING
)
USING DELTA
CLUSTER BY (plan_itinerary_id, id)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

-- Bảng users
CREATE TABLE IF NOT EXISTS lh_vv_bronze.partner.users (
    id STRING NOT NULL,
    created_at BIGINT,
    updated_at BIGINT,
    deactived_at BIGINT,
    deleted_at BIGINT,
    status STRING,
    is_guest BOOLEAN,
    iam_user_id STRING,
    avatar STRING,
    avatar_blur_hash STRING,
    first_name STRING,
    last_name STRING,
    birth_date BIGINT,
    email STRING,
    email_unverified STRING,
    is_email_locked BOOLEAN,
    dial_code STRING,
    phone STRING,
    dial_code_unverified STRING,
    phone_unverified STRING,
    is_phone_locked BOOLEAN,
    gender STRING,
    nationality STRING,
    citizen_id STRING,
    citizen_id_type STRING,
    metadata STRING,
    lang STRING,
    deleted BOOLEAN,
    _ingested_at TIMESTAMP, 
    _source_db STRING
)
USING DELTA
CLUSTER BY (id)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);


-- =============================================================================
-- 1. travel_profiles
-- =============================================================================
MERGE INTO lh_vv_bronze.partner.travel_profiles AS t
USING (
  SELECT id, created_at, updated_at, is_active, user_id, persona, travel_dna,
         travel_styles, dietary_preferences, accommodation_amenities_preferences,
         metadata, deleted, _ingested_at, _source_db
  FROM (
    SELECT
      get_json_object(after, '$.id')                                              AS id,
      CAST(get_json_object(after, '$.created_at')                    AS BIGINT)   AS created_at,
      CAST(get_json_object(after, '$.updated_at')                    AS BIGINT)   AS updated_at,
      CAST(get_json_object(after, '$.is_active')                     AS BOOLEAN)  AS is_active,
      get_json_object(after, '$.user_id')                                          AS user_id,
      get_json_object(after, '$.persona')                                          AS persona,
      get_json_object(after, '$.travel_dna')                                       AS travel_dna,
      get_json_object(after, '$.travel_styles')                                    AS travel_styles,
      get_json_object(after, '$.dietary_preferences')                              AS dietary_preferences,
      get_json_object(after, '$.accommodation_amenities_preferences')              AS accommodation_amenities_preferences,
      get_json_object(after, '$.metadata')                                         AS metadata,
      FALSE                                                                         AS deleted,
      current_timestamp()                                                           AS _ingested_at,
      get_json_object(source, '$.db')                                              AS _source_db,
      ROW_NUMBER() OVER (
        PARTITION BY get_json_object(after, '$.id')
        ORDER BY CAST(get_json_object(source, '$.ts_ms') AS BIGINT) DESC
      ) AS rn
    FROM lh_vv_bronze.dbo.userprofile_raw_data
    WHERE get_json_object(source, '$.table') = 'travel_profiles'
      AND op IN ('c', 'r', 'u')
  )
  WHERE rn = 1
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO lh_vv_bronze.partner.travel_profiles AS t
USING (
  SELECT DISTINCT get_json_object(before, '$.id') AS id
  FROM lh_vv_bronze.dbo.userprofile_raw_data
  WHERE get_json_object(source, '$.table') = 'travel_profiles'
    AND op = 'd'
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.deleted = TRUE;


-- =============================================================================
-- 2. user_plans
-- =============================================================================
MERGE INTO lh_vv_bronze.partner.user_plans AS t
USING (
  SELECT id, creator_id, title, start_date, end_date, estimated_total_person,
         estimated_total_budget, estimated_budgets, currency, weather_forecast,
         traffic_suggestions, trip_id, total_days, share_code, created_at,
         updated_at, metadata, deleted, _ingested_at, _source_db
  FROM (
    SELECT
      get_json_object(after, '$.id')                                              AS id,
      get_json_object(after, '$.creator_id')                                      AS creator_id,
      get_json_object(after, '$.title')                                           AS title,
      CAST(get_json_object(after, '$.start_date')                    AS BIGINT)   AS start_date,
      CAST(get_json_object(after, '$.end_date')                      AS BIGINT)   AS end_date,
      CAST(get_json_object(after, '$.estimated_total_person')        AS INT)      AS estimated_total_person,
      get_json_object(after, '$.estimated_total_budget')                          AS estimated_total_budget,
      get_json_object(after, '$.estimated_budgets')                               AS estimated_budgets,
      get_json_object(after, '$.currency')                                        AS currency,
      get_json_object(after, '$.weather_forecast')                                AS weather_forecast,
      get_json_object(after, '$.traffic_suggestions')                             AS traffic_suggestions,
      get_json_object(after, '$.trip_id')                                         AS trip_id,
      CAST(get_json_object(after, '$.total_days')                    AS INT)      AS total_days,
      get_json_object(after, '$.share_code')                                      AS share_code,
      CAST(get_json_object(after, '$.created_at')                    AS BIGINT)   AS created_at,
      CAST(get_json_object(after, '$.updated_at')                    AS BIGINT)   AS updated_at,
      get_json_object(after, '$.metadata')                                         AS metadata,
      FALSE                                                                         AS deleted,
      current_timestamp()                                                           AS _ingested_at,
      get_json_object(source, '$.db')                                              AS _source_db,
      ROW_NUMBER() OVER (
        PARTITION BY get_json_object(after, '$.id')
        ORDER BY CAST(get_json_object(source, '$.ts_ms') AS BIGINT) DESC
      ) AS rn
    FROM lh_vv_bronze.dbo.userprofile_raw_data
    WHERE get_json_object(source, '$.table') = 'user_plans'
      AND op IN ('c', 'r', 'u')
  )
  WHERE rn = 1
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO lh_vv_bronze.partner.user_plans AS t
USING (
  SELECT DISTINCT get_json_object(before, '$.id') AS id
  FROM lh_vv_bronze.dbo.userprofile_raw_data
  WHERE get_json_object(source, '$.table') = 'user_plans'
    AND op = 'd'
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.deleted = TRUE;


-- =============================================================================
-- 3. user_plan_itineraries
-- =============================================================================
MERGE INTO lh_vv_bronze.partner.user_plan_itineraries AS t
USING (
  SELECT id, plan_id, day_number, `date`, day_type, weather_forecast,
         traffic_suggestion, created_at, updated_at, metadata, deleted,
         _ingested_at, _source_db
  FROM (
    SELECT
      get_json_object(after, '$.id')                                              AS id,
      get_json_object(after, '$.plan_id')                                         AS plan_id,
      CAST(get_json_object(after, '$.day_number')                    AS INT)      AS day_number,
      CAST(get_json_object(after, '$.date')                          AS BIGINT)   AS `date`,
      get_json_object(after, '$.day_type')                                        AS day_type,
      get_json_object(after, '$.weather_forecast')                                AS weather_forecast,
      get_json_object(after, '$.traffic_suggestion')                              AS traffic_suggestion,
      CAST(get_json_object(after, '$.created_at')                    AS BIGINT)   AS created_at,
      CAST(get_json_object(after, '$.updated_at')                    AS BIGINT)   AS updated_at,
      get_json_object(after, '$.metadata')                                         AS metadata,
      FALSE                                                                         AS deleted,
      current_timestamp()                                                           AS _ingested_at,
      get_json_object(source, '$.db')                                              AS _source_db,
      ROW_NUMBER() OVER (
        PARTITION BY get_json_object(after, '$.id')
        ORDER BY CAST(get_json_object(source, '$.ts_ms') AS BIGINT) DESC
      ) AS rn
    FROM lh_vv_bronze.dbo.userprofile_raw_data
    WHERE get_json_object(source, '$.table') = 'user_plan_itineraries'
      AND op IN ('c', 'r', 'u')
  )
  WHERE rn = 1
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO lh_vv_bronze.partner.user_plan_itineraries AS t
USING (
  SELECT DISTINCT get_json_object(before, '$.id') AS id
  FROM lh_vv_bronze.dbo.userprofile_raw_data
  WHERE get_json_object(source, '$.table') = 'user_plan_itineraries'
    AND op = 'd'
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.deleted = TRUE;


-- =============================================================================
-- 4. user_plan_itinerary_items
-- =============================================================================
MERGE INTO lh_vv_bronze.partner.user_plan_itinerary_items AS t
USING (
  SELECT id, plan_itinerary_id, `rank`, itinerary_type, item_id, item_metadata,
         suggestion_note, category_key, selection_reason, created_at, updated_at,
         metadata, deleted, _ingested_at, _source_db
  FROM (
    SELECT
      get_json_object(after, '$.id')                                              AS id,
      get_json_object(after, '$.plan_itinerary_id')                               AS plan_itinerary_id,
      CAST(get_json_object(after, '$.rank')                          AS INT)      AS `rank`,
      get_json_object(after, '$.itinerary_type')                                  AS itinerary_type,
      get_json_object(after, '$.item_id')                                         AS item_id,
      get_json_object(after, '$.item_metadata')                                    AS item_metadata,
      get_json_object(after, '$.suggestion_note')                                 AS suggestion_note,
      get_json_object(after, '$.category_key')                                    AS category_key,
      get_json_object(after, '$.selection_reason')                                AS selection_reason,
      CAST(get_json_object(after, '$.created_at')                    AS BIGINT)   AS created_at,
      CAST(get_json_object(after, '$.updated_at')                    AS BIGINT)   AS updated_at,
      get_json_object(after, '$.metadata')                                         AS metadata,
      FALSE                                                                         AS deleted,
      current_timestamp()                                                           AS _ingested_at,
      get_json_object(source, '$.db')                                              AS _source_db,
      ROW_NUMBER() OVER (
        PARTITION BY get_json_object(after, '$.id')
        ORDER BY CAST(get_json_object(source, '$.ts_ms') AS BIGINT) DESC
      ) AS rn
    FROM lh_vv_bronze.dbo.userprofile_raw_data
    WHERE get_json_object(source, '$.table') = 'user_plan_itinerary_items'
      AND op IN ('c', 'r', 'u')
  )
  WHERE rn = 1
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO lh_vv_bronze.partner.user_plan_itinerary_items AS t
USING (
  SELECT DISTINCT get_json_object(before, '$.id') AS id
  FROM lh_vv_bronze.dbo.userprofile_raw_data
  WHERE get_json_object(source, '$.table') = 'user_plan_itinerary_items'
    AND op = 'd'
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.deleted = TRUE;


-- =============================================================================
-- 5. users
-- =============================================================================
MERGE INTO lh_vv_bronze.partner.users AS t
USING (
  SELECT id, created_at, updated_at, deactived_at, deleted_at, status, is_guest,
         iam_user_id, avatar, avatar_blur_hash, first_name, last_name, birth_date,
         email, email_unverified, is_email_locked, dial_code, phone,
         dial_code_unverified, phone_unverified, is_phone_locked, gender,
         nationality, citizen_id, citizen_id_type, metadata, lang, deleted,
         _ingested_at, _source_db
  FROM (
    SELECT
      get_json_object(after, '$.id')                                              AS id,
      CAST(get_json_object(after, '$.created_at')                    AS BIGINT)   AS created_at,
      CAST(get_json_object(after, '$.updated_at')                    AS BIGINT)   AS updated_at,
      CAST(get_json_object(after, '$.deactived_at')                  AS BIGINT)   AS deactived_at,
      CAST(get_json_object(after, '$.deleted_at')                    AS BIGINT)   AS deleted_at,
      get_json_object(after, '$.status')                                           AS status,
      CAST(get_json_object(after, '$.is_guest')                     AS BOOLEAN)  AS is_guest,
      get_json_object(after, '$.iam_user_id')                                      AS iam_user_id,
      get_json_object(after, '$.avatar')                                           AS avatar,
      get_json_object(after, '$.avatar_blur_hash')                                 AS avatar_blur_hash,
      get_json_object(after, '$.first_name')                                       AS first_name,
      get_json_object(after, '$.last_name')                                        AS last_name,
      CAST(get_json_object(after, '$.birth_date')                    AS BIGINT)   AS birth_date,
      get_json_object(after, '$.email')                                            AS email,
      get_json_object(after, '$.email_unverified')                                 AS email_unverified,
      CAST(get_json_object(after, '$.is_email_locked')               AS BOOLEAN)  AS is_email_locked,
      get_json_object(after, '$.dial_code')                                        AS dial_code,
      get_json_object(after, '$.phone')                                            AS phone,
      get_json_object(after, '$.dial_code_unverified')                            AS dial_code_unverified,
      get_json_object(after, '$.phone_unverified')                                AS phone_unverified,
      CAST(get_json_object(after, '$.is_phone_locked')              AS BOOLEAN)  AS is_phone_locked,
      get_json_object(after, '$.gender')                                           AS gender,
      get_json_object(after, '$.nationality')                                      AS nationality,
      get_json_object(after, '$.citizen_id')                                       AS citizen_id,
      get_json_object(after, '$.citizen_id_type')                                  AS citizen_id_type,
      get_json_object(after, '$.metadata')                                         AS metadata,
      get_json_object(after, '$.lang')                                             AS lang,
      FALSE                                                                         AS deleted,
      current_timestamp()                                                           AS _ingested_at,
      get_json_object(source, '$.db')                                              AS _source_db,
      ROW_NUMBER() OVER (
        PARTITION BY get_json_object(after, '$.id')
        ORDER BY CAST(get_json_object(source, '$.ts_ms') AS BIGINT) DESC
      ) AS rn
    FROM lh_vv_bronze.dbo.userprofile_raw_data
    WHERE get_json_object(source, '$.table') = 'users'
      AND op IN ('c', 'r', 'u')
  )
  WHERE rn = 1
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO lh_vv_bronze.partner.users AS t
USING (
  SELECT DISTINCT get_json_object(before, '$.id') AS id
  FROM lh_vv_bronze.dbo.userprofile_raw_data
  WHERE get_json_object(source, '$.table') = 'users'
    AND op = 'd'
) AS s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.deleted = TRUE;



StatementMeta(, b0f20a82-400e-4a06-93c5-696fe66e90a0, 19, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 1 rows and 4 fields>